In [1]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

E0000 00:00:1748969744.810388   45811 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748969744.815490   45811 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748969744.830939   45811 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748969744.830970   45811 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748969744.830972   45811 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748969744.830974   45811 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [3]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
#dev = tf.config.list_physical_devices()
#print('Available Devices : ', dev)
#tf.config.set_visible_devices(dev[0], 'CPU')
#tf.config.set_visible_devices(dev[1], 'GPU')

# Chapter 5: Fundamentals of Machine Learning

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

After the three practical examples from the previous unit, you should be starting to feel
familiar with how to approach classification and regression problems using neural networks.

You've also experienced some discussion on the central problem of machine learning: **overfitting**.

In this unit we will attempt to improve and formalize some of these intuitions about
machine learning into a solid conceptual framework.  Being good at building machine learning
models  means that you know how to accurately evaluate models, and how to balance between
training and generalization.


## 5.2 Evaluating Machine Learning Models

You can only control what you can observe.

### 5.2.1 Training, validation and test sets

You train on the training data and evaluate your model while fitting it and while
comparing alternative models on the validation data. Once your model is ready for
prime time, you test it one final time on the test data, which is meant to be as
similar as possible to production data. Then you can deploy the model in production.

Developing a model always involves tuning its configuration.  You do this tuning by using
a feedback signal: the performance of the model on the validation data.  

Tuning the configuration of a model or multiple modles based on its performance
on the validation set can quickly result in *overfitting to the validation set*,
even though your model is never directly trained on it.  

But we care about performance on completely new data, not test data, and not the
validation data we are using to try and optimize the model architecture.  This is
why a completely different, never-before-seen dataset is needed to evaluate a
final model, the test dataset.

#### Simple holdout validation

In order to prevent information leaks, you shouldn’t tune your model based on the test set, and therefore you
should also reserve a validation set.

A simple implementaiton of holdout validation can be done as follows (this is random data, just for illustration).

```python
# data with 100,000 samples of 50 features for illustration
num_validation_samples = 10000
all_train_datadata = np.random.random((100000, 50))

# shuffling the data is usually appropriate, it could be sorted or orderd when you get it
# default behavior of np.random.shuffle on a tensor is to only shuffle the 0-th (sample) dimension
np.random.shuffle(all_train_data)


# define the validation set, slize the first 0-num_validation_samples from the data for validation
validation_data = all_train_data[:num_validation_samples]

# define the training set, use remaining data after split
train_data = all_train_datadata[num_validation_samples:]

# train a model on the training data and evaluatte it on the validation data
model = get_model()
model.fit(training_data, ...)
validation_score = model.evaluate(validation_data, ...)

# at this point you look at validation_score performance on validation data, and tune your
# model and model parameters.  Then retrain it and evaluate again, repeat
# as necessary until satisfied with performance

# once you've tuned your hyperparameters, its common to train your final
# model from stratch on all non-test data available
model = get_model()
model.fit(all_train_data, ...)
test_score = model.evaluate(test_data, ...)
```

#### K-fold validation

With this approach, you split your data into `K` partitions of equal size. For each partition
`i`, train a model on the remaining `K - 1` partitions, and evaluate it on partition `i`.
Your final score is then the averages of the K scores obtained.

This method is helpful when the performance of your model shows significant variance based on your train/test
split.

A simple pseudo code implementaiton by hand of K-fold validation looks like the following

```python
# data with 100,000 samples of 50 features for illustration
all_train_datadata = np.random.random((100000, 50))

# number of folds to divide data into
k = 3

# integer division will divide data into as equal sized partitions as possible, though
# last fold can have a few extra with this method
num_validation_samples = len(data) // k
```

In [9]:
# for instance, the size of the folds when we have around 100,000 samples would be
print(99999 // 3)  # all folds exactly equal in size
print(100000 // 3) # last fold has 1 extra sample
print(100001 // 3) # last fold has 2 extra samples
print(100002 // 3) # all folds exactly equal again

33333
33333
33333
33334


```python
# always a good idea to shuffle data in case it is sorted or ordered
np.random.shuffle(all_train_data)

# keep results for each fold validation to be reported and averged at end
validation_scores = []

# in k-fold validation, we perform a number of separate trainings of the model equal to k
for fold in range(k):
    # selects the validation-data partition from all of the data
    validation_data = data[num_validation_samples * fold: num_validation_samples * (fold + 1)]

    # combine remainder of the data for the training set
    training_data = np.concatenate(
        data[:num_validation_samples * fold],
        data[num_validation_samples * (fold + 1):]
    )

    # make sure you create a brand new instance of a model, can be easy to continue
    # training a previously trained model using keras if don't specificly create new model
    # with randomly initialized weights
    model = get_model()
    model.fit(training_data, ...)

    # use validation data to evaluate this fold
    validation_score = model.evaluate(validation_data, ...)
    validation_scores.append(validation_score)

# after k folds of training and evaluation on held back validation data,
# a better measure of generalization is the average of the individual validation scores
validation_score = np.average(validation_scores)

# train a new final model on all non-test data available and evaluate with the final test data
model = get_model()
model.fit(all_train_data, ...)
test_score = model.evaluate(test_data, ...)
```

#### Iterated K-fold validation with shuffling

This one is for situations in which you have relatively little data available and you need
to evaluate your model as precisely as possible. It consists of applying K-fold validation
`P` times, shuffling the data every time before splitting `K` ways.  The final score is the 
average of the `P` runs of K-fold validation.  Note that you end up training and
evaluation `P * K` models.

In essence, if you turned the K-fold validation code into a function that you could call, that would return
the average validation score as a result, and the function shuffled the data passed in each
time before dividing into folds, you could call this function `P` times to perform
iterated K-fold validation.

### 5.2.2 Beating a common-sense baseline

Before you start working with a dataset, you should always pick
a trivial baseline that you’ll try to beat.

### 5.2.3 Things to keep in mind about model evaluation

- **Data representativeness** You want both your training set and test set to be representative of the data at hand.
  For this reason you usually should at least **randomly shuffle** your data before splitting it into training and test sets.
- **The arrow of time** On the other hand, f you are doing time series prediction, you should **NOT** randomly shuffle your data
  before splitting it, this will cause **temporal leak**.  Instead always make sure all data in test set is *posterior* to the
  data in the training set.
- **Redundancy in your data** Be careful of repeated data points appearing multiple times in a dataset (not uncommon).  If you
  split a repeated sample and end up with it in the train and test set, you'll be leaking information and essentially testing
  with part of the data you trained with.

## Summary 

To recap, here is what you should remember about how to evaluate machine learning models effectively.

<font color='blue'>

- 